# Models
[Previous Notebook](preprocessing.ipynb) | [**Next Notebook** (TBD)]()

To begin this section, i would like to do some inital analysis of certain models discussed in the course and why they may or may not be a good fit for the problem

## Instance and tree based
### K Nearest Neighbours
As said before, the dataset is very imbalanced (~97% bias towards not depressed). Because of this, I think that voting by neighbourhood majority will not be a very effective means of predicting the target, especially if the individual features have a lot of varience, which i saw evidence of through the `LogisticRegression` tests of the previous notebook (~0.23 `std`).

### Decision Tree
Unlike `KNN`, A `Decision Tree` has the potential to be a very good model for this problem. With a suitible `max_depth` and split criteria, a decision tree can uncover more complex linear patterns in a dataset and is much more fitted for imbalanced data than KNN.

## Ensembles
### Random Forest
Although `Random Forest` is built on the back of decision trees, I do not see it being an effective model for this problem. By definition, `Random Forest` chooses a random subset of features for each split while constructing the tree for each bootstrap sample and then aggregate by voting. Fro my understanding of the problem and due to the poor balance of the dataset, i dont think aggregation (voting) will be a fair means of resolving a prediction. 

### Gradient Boost
Building on the predicted effectiveness of a `Decision Tree`, I think `Gradient Boost` will be a better ensemble method for increasing model performance, as it incrementally cancels the errors of the model using the samples loss gradient. From my own intuition, keeping in mind the imbalanced dataset and the logic of the problem, incrementally adjusting the error of decisions in the tree makes more sense than randomly choosing features for each split and aggregating the results at the end.

# To begin...
I will allow importing my modular scripts from [src/](/src/) and split my dataset into training and test sets to avoid leakage.

In [35]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
import numpy as np
import environment

In [36]:
from get_data_splits import get_data_splits

splits = get_data_splits()
X_train, X_test, y_train, y_test = splits

# KNN

In [37]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from preprocessor import preprocessor
from grid_cross_validation import grid_cross_validation

knn = make_pipeline(preprocessor, StandardScaler(), KNeighborsClassifier())

cv, best_estimator, best_params = grid_cross_validation(knn, X_train, y_train, param_grid={
    'kneighborsclassifier__n_neighbors': [3, 5, 7, 11, 21],
    'kneighborsclassifier__weights': ['uniform', 'distance'],
    'kneighborsclassifier__p': [1, 2, 3]
})

knn_cv = cv
display(best_params)
print("Best f1 score: ", cv.best_score_)

,Best Parameter
n_neighbors,3
p,3
weights,uniform


Best f1 score:  0.05714285714285714


In [ ]:
from run_test import run_test

results = run_test(best_estimator, splits)
display(results['confusion_matrix'])
print(results['classification_report'])
knn_f1_score = results['f1_score']

,predicted not depressed,predicted depressed
actual not depressed,350,1
actual depressed,8,1


Classification Report
               precision    recall  f1-score   support

not depressed       0.98      1.00      0.99       351
    depressed       0.50      0.11      0.18         9

     accuracy                           0.97       360
    macro avg       0.74      0.55      0.58       360
 weighted avg       0.97      0.97      0.97       360



### Interpretation of results

As you can see, `KNN` performs absolutely horribly for my problem, with `0`'s for all positive metrics. This makes sense as the dataset is so imbalanced and the datapoints likely do not lie very close in feature space. Additionally, the added dimensions due to onehot encoding skew the distance values.

# Decision Tree

In [39]:
from sklearn.tree import DecisionTreeClassifier

decisionTree = make_pipeline(preprocessor, DecisionTreeClassifier(criterion='entropy', random_state=environment.Random_State))

cv, best_estimator, best_params = grid_cross_validation(decisionTree, X_train, y_train, param_grid={
    'decisiontreeclassifier__max_depth': [1, 3, 5, 10],
})

decisionTree_cv = cv
display(best_params)
print("Best f1 score: ", cv.best_score_)

,Best Parameter
max_depth,5


Best f1 score:  0.9111111111111111


In [40]:
from run_test import run_test

results = run_test(best_estimator, splits)
display(results['confusion_matrix'])
print(results['classification_report'])

,predicted not depressed,predicted depressed
actual not depressed,351,0
actual depressed,1,8


Classification Report
               precision    recall  f1-score   support

not depressed       1.00      1.00      1.00       351
    depressed       1.00      0.89      0.94         9

     accuracy                           1.00       360
    macro avg       1.00      0.94      0.97       360
 weighted avg       1.00      1.00      1.00       360



# Random Forest

In [41]:
from sklearn.ensemble import RandomForestClassifier

randForest = make_pipeline(preprocessor, RandomForestClassifier(criterion='entropy', n_jobs=-1, random_state=environment.Random_State))

cv, best_estimator, best_params = grid_cross_validation(randForest, X_train, y_train, param_grid={
    'randomforestclassifier__max_depth': [1, 3, 5, 10, 15, 20],
    'randomforestclassifier__n_estimators': [10, 50, 100],
})

randForest_cv = cv
display(best_params)
print("Best f1 score: ", cv.best_score_)

,Best Parameter
max_depth,10
n_estimators,50


Best f1 score:  0.21333333333333332


In [42]:
results = run_test(best_estimator, splits)
display(results['confusion_matrix'])
print(results['classification_report'])

,predicted not depressed,predicted depressed
actual not depressed,351,0
actual depressed,5,4


Classification Report
               precision    recall  f1-score   support

not depressed       0.99      1.00      0.99       351
    depressed       1.00      0.44      0.62         9

     accuracy                           0.99       360
    macro avg       0.99      0.72      0.80       360
 weighted avg       0.99      0.99      0.98       360



# Gradient Boost

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

GBoost = make_pipeline(preprocessor, GradientBoostingClassifier(random_state=environment.Random_State))

cv, best_estimator, best_params = grid_cross_validation(GBoost, X_train, y_train, param_grid={
    'gradientboostingclassifier__max_depth': [3, 5, 7, 10],
    'gradientboostingclassifier__n_estimators': [10, 100, 500, 1000],
    'gradientboostingclassifier__learning_rate': [0.01, 0.1, 0.2, 0.5, 1]
})

GBoost_cv = cv
display(best_params)
print("Best f1 score: ", cv.best_score_)

In [ ]:
results = run_test(best_estimator, splits)
display(results['confusion_matrix'])
print(results['classification_report'])

,predicted not depressed,predicted depressed
actual not depressed,351,0
actual depressed,1,8


Classification Report
               precision    recall  f1-score   support

not depressed       1.00      1.00      1.00       351
    depressed       1.00      0.89      0.94         9

     accuracy                           1.00       360
    macro avg       1.00      0.94      0.97       360
 weighted avg       1.00      1.00      1.00       360



# To conclude this section...


In [ ]:
display(pd.DataFrame([
    knn_cv.best_score_,
    randForest_cv.best_score_,
    np.float64(0.545455),
    decisionTree_cv.best_score_,
    GBoost_cv.best_score_
], index=['KNN', 'Random Forest', 'Logistic Regression (baseline)', 'Decision Tree', 'Gradient Boosting'], columns=['Best f1 score']))

,Best f1 score
KNN,0.057143
Random Forest,0.213333
Logistic Regression (baseline),0.545455
Decision Tree,0.911111
Gradient Boosting,0.911111
